In [1]:
import pandas as pd
import numpy as np

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Load output from Phase 11
inv_path = '../data/processed/inventory_analysis_table.csv'
df = pd.read_csv(inv_path)

print(f"Loaded {len(df):,} SKU records from inventory analytics pipeline.")

Loaded 1,676 SKU records from inventory analytics pipeline.


In [3]:
# Service Level Configuration
Z_SCORE = 1.65          # 95% Service Level
LEAD_TIME_WEEKS = 2     # 2-Week Replenishment Lead Time
TARGET_COVERAGE_WKS = 4 # Target Coverage Horizon

# Assign missing column to DataFrame
df['Lead_Time_Weeks'] = LEAD_TIME_WEEKS

# 1. Lead Time Demand
df['Lead_Time_Demand'] = (df['Recent_Forecast_Demand'] * LEAD_TIME_WEEKS).round(2)

# 2. Safety Stock Calculation
df['Safety_Stock'] = np.ceil(Z_SCORE * df['Demand_Std_Dev'] * np.sqrt(LEAD_TIME_WEEKS))

# 3. Reorder Point (ROP)
df['Reorder_Point'] = df['Lead_Time_Demand'] + df['Safety_Stock']

# 4. Target Stock Horizon & Recommended Reorder Quantity
df['Target_Stock'] = df['Recent_Forecast_Demand'] * TARGET_COVERAGE_WKS
df['Inventory_Position'] = df['Simulated_Current_Stock']

# If Inventory Position <= ROP, Reorder Quantity = Target_Stock - Inventory_Position, else 0
df['Reorder_Quantity'] = np.where(
    df['Inventory_Position'] <= df['Reorder_Point'],
    np.maximum(0, np.ceil(df['Target_Stock'] - df['Inventory_Position'])),
    0
)

# 5. Business Recommended Action Flag
def assign_action(row):
    if row['Inventory_Position'] <= row['Reorder_Point']:
        return 'REORDER IMMEDIATELY'
    elif row['Inventory_Position'] <= (row['Reorder_Point'] + row['Safety_Stock']):
        return 'MONITOR CLOSELY'
    elif row['Inventory_Position'] > (row['Reorder_Point'] * 2.5):
        return 'OVERSTOCKED - HALT PURCHASING'
    else:
        return 'STOCK LEVEL OPTIMAL'

df['Recommended_Action'] = df.apply(assign_action, axis=1)

# Format Final Decision Recommendation Table
rec_cols = [
    'StockCode', 'Recent_Forecast_Demand', 'Lead_Time_Weeks', 
    'Safety_Stock', 'Reorder_Point', 'Inventory_Position', 
    'Reorder_Quantity', 'Recommended_Action'
]

recommendation_table = df[rec_cols].copy()
recommendation_table.rename(columns={'Recent_Forecast_Demand': 'Forecast_Demand'}, inplace=True)

print("=== REORDER RECOMMENDATION SUMMARY ===")
print(recommendation_table['Recommended_Action'].value_counts())
display(recommendation_table.head(10))

=== REORDER RECOMMENDATION SUMMARY ===
Recommended_Action
REORDER IMMEDIATELY              697
MONITOR CLOSELY                  623
STOCK LEVEL OPTIMAL              332
OVERSTOCKED - HALT PURCHASING     24
Name: count, dtype: int64


,StockCode,Forecast_Demand,Lead_Time_Weeks,Safety_Stock,Reorder_Point,Inventory_Position,Reorder_Quantity,Recommended_Action
0,10002,52.75,2,363.00,468.50,573.00,0.00,MONITOR CLOSELY
1,10120,13.75,2,49.00,76.50,137.00,0.00,STOCK LEVEL OPTIMAL
2,10125,22.00,2,82.00,126.00,206.00,0.00,MONITOR CLOSELY
3,10133,163.75,2,153.00,480.50,326.00,329.00,REORDER IMMEDIATELY
4,10135,34.50,2,210.00,279.00,211.00,0.00,REORDER IMMEDIATELY
5,11001,24.50,2,178.00,227.00,193.00,0.00,REORDER IMMEDIATELY
6,15036,124.50,2,1182.00,1431.00,1164.00,0.00,REORDER IMMEDIATELY
7,15039,38.00,2,205.00,281.00,507.00,0.00,STOCK LEVEL OPTIMAL
8,15044C,3.00,2,35.00,41.00,72.00,0.00,MONITOR CLOSELY
9,15044D,6.25,2,60.00,72.50,138.00,0.00,STOCK LEVEL OPTIMAL


In [ ]:
# Select top SKU for manual formula verification
sample_sku = recommendation_table.iloc[0]

sku_code = sample_sku['StockCode']
forecast_d = sample_sku['Forecast_Demand']
std_dev = df.loc[df['StockCode'] == sku_code, 'Demand_Std_Dev'].values[0]
inv_pos = sample_sku['Inventory_Position']

# Manual step calculation
manual_ltd = forecast_d * 2
manual_ss = np.ceil(1.65 * std_dev * np.sqrt(2))
manual_rop = manual_ltd + manual_ss
manual_target = forecast_d * 4
manual_rq = np.ceil(manual_target - inv_pos) if inv_pos <= manual_rop else 0

print(f"--- MANUAL CALCULATION AUDIT (SKU: {sku_code}) ---")
print(f"Forecast Demand (d):     {forecast_d:.2f}")
print(f"Demand Std Dev (sigma):  {std_dev:.2f}")
print(f"Manual LTD (d * 2):      {manual_ltd:.2f} | Code Output: {sample_sku['Forecast_Demand']*2:.2f}")
print(f"Manual SS (1.65*s*v2):   {manual_ss:.0f} | Code Output: {sample_sku['Safety_Stock']:.0f}")
print(f"Manual ROP (LTD + SS):   {manual_rop:.2f} | Code Output: {sample_sku['Reorder_Point']:.2f}")
print(f"Manual Reorder Quantity: {manual_rq:.0f} | Code Output: {sample_sku['Reorder_Quantity']:.0f}")

assert round(manual_ss) == round(sample_sku['Safety_Stock']), "Safety Stock Math Mismatch!"
assert round(manual_rop) == round(sample_sku['Reorder_Point']), "ROP Math Mismatch!"
print("\n[SUCCESS] Manual calculation check matches Python code output perfectly.")